# Neural Network converter (quantify)

This notebook is meant to show a step by step guide for several Neural Network conversion modes (quantization). This conversions will simplify the algorithm so it needs less complex calculations, and by doing so it can be ported successfully into a micro-controller.

## Imports and definitions

In [ ]:
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

import sys

# More imports
import matplotlib.pyplot as plt #for plotting raw data
import os #for system folders and files management
import pathlib #for system folders and files management
import pandas as pd #for Dataframe handling

from IPython.display import display # for folder selection dialog
from ipyfilechooser import FileChooser # for folder selection dialog

import numpy as np
import math
import csv

#for metrics and plots
import sklearn.model_selection
from sklearn.utils import class_weight
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report
import seaborn as sns #for plotting confusion matrix

# End

## Loading model

In [ ]:
#Select model:

# Create and display a FileChooser widget
fc_model = FileChooser('./')
#fc_test.show_only_dirs=False
print("------------------------------\nSELECT TRAINED MODEL\n------------------------------")
display(fc_model)
print("------------------------------\n")

In [ ]:
# Loading model

if fc_model.selected_path==None:
    print("No model specified!!")
    #model = tf.keras.models.load_model(os.path.join(sensor_model_folder, best_model_lstm_name))
else:
    model = tf.keras.models.load_model(fc_model.selected)
    pred = model.predict(TestDataX)
    print("--------\nDone!")

## Conversion

### Mode 1: default post-training quantization

In [ ]:
# Creamos el conversor
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Creamos el modelo de TFLite a partir del conversor
tflite_model = converter.convert()
     
tflite_model_path = "{}_q1.h5".format(fc_model.selected)

with open(tflite_model_path, 'wb') as f:
    f.write(tflite_model)
     


### Mode 2: 8 bit post-training quantization

In [ ]:
# Creamos el conversor
converter = tf.lite.TFLiteConverter.from_keras_model(model)
# Variamos la configuración, con esta sentencia configuramos el conversor para que cuantice a 8bits
converter.optimizations = [tf.lite.Optimize.DEFAULT]
# Creamos el modelo de TFLite a partir del conversor
tflite_quant_model = converter.convert()

# Load the TFLite model and allocate tensors.
interpreter = tf.lite.Interpreter(model_content=tflite_quant_model)
interpreter.allocate_tensors()

# Get input and output tensors.
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()


# print('input_details: ', input_details)
# print('output_details: ', output_details)


# Test the model on random input data.
input_shape = input_details[0]['shape']

# Es necesario que tanto la forma como el tipo de los datos que componen el 
# dataset sean los que admita el modelo TFLite configurado. 
x_test_type_float32 = x_test.astype(np.float32)
#print('Tipo de los datos tras la conversión: ', type(x_test_type_float32[0,0,0]))
input_data = x_test_type_float32

#print('shape', np.shape(input_data))
interpreter.resize_tensor_input(input_details[0]['index'],np.shape(input_data))
interpreter.allocate_tensors()

interpreter.set_tensor(input_details[0]['index'], input_data)

interpreter.invoke()

# The function `get_tensor()` returns a copy of the tensor data.
# Use `tensor()` in order to get a pointer to the tensor.
output_data = interpreter.get_tensor(output_details[0]['index'])
print(output_data)

### Mode 3: full integer post-training quantization